# 🫀 ChagaSight — Fold Training (Colab)

## Run order every session
| Cell | Purpose | Run when |
|------|---------|----------|
| 1 | Mount Drive + GPU check | **Every session** |
| 2 | Install packages | **Every session** |
| 3 | Stage signals to local disk | **Every session** (~3–8 min) |
| 4 | Configuration | **Every session** |
| 5 | Imports | **Every session** |
| 6 | GPU memory check | Every session |
| 7 | Dataloaders | Every session |
| 8 | Model + pretrained weights | Every session |
| 9 | Trainer | Every session |
| 10 | **TRAIN** | Every session — auto-resumes |
| 11 | Save results + plot | After training |
| 12 | Checkpoint verification | After training |

## Key differences vs local notebook
- **`PHASE2_GRAD_ACCUM = 4`** → eff.batch = 64 (T4 has 15 GB, no need to halve)
- **`PHASE2_ITERATIONS = 12000`** (not 24000 — accum=4 already matches paper)
- Signals staged to `/content/signals/` to avoid Drive FUSE latency
- Images read directly from Drive (individual file opens work fine — no directory listing)
- Keepalive thread prevents Colab idle disconnect
- Training auto-saves to Drive every checkpoint


## Cell 1 — Mount Drive & GPU Check

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import subprocess, torch

# GPU info
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:600])
print(f'PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1e9
    print(f'GPU:  {props.name}')
    print(f'VRAM: {vram:.1f} GB')
    torch.backends.cudnn.benchmark = True
    print('cudnn.benchmark = True  (+5-10% speed)')

    if vram < 10:
        print('⚠️  Less than 10 GB — go to Runtime > Change runtime type > T4 GPU')
    elif vram >= 30:
        print('✓ A100 detected — can use BATCH_SIZE=32 in Cell 4 for 2× speedup')
    else:
        print('✓ T4 detected — config below is optimised for this GPU')
else:
    print('❌ No GPU — Runtime > Change runtime type > T4 GPU')


## Cell 2 — Install Packages

In [ ]:
# Install packages not pre-installed in Colab
import subprocess, sys

pkgs = ['wfdb', 'scikit-learn']  # torch, numpy, pandas already in Colab

for pkg in pkgs:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✓' if result.returncode == 0 else '✗'
    print(f'{status}  {pkg}')

print('\nPackages ready.')


## Cell 3 — Stage Signals to Local Disk

**Why**: Drive FUSE latency (~5ms per file) would add ~90ms overhead per 16-sample
batch just for signal reads. Staging 14 GB of signals to `/content/signals/` once
per session cuts this to ~0.1ms per file.

**Images** are NOT staged (39 GB too large). Direct Drive opens work fine since
we open files by exact path, never listing the directory.

**Time**: ~3–8 min first run, ~1 min on reconnect (skips existing files).


In [ ]:
import pandas as pd, shutil, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# ── Config (adjust if your Drive path differs) ────────────────────────────
DRIVE_ROOT    = '/content/drive/MyDrive/ChagaSight'
DRIVE_SIG_DIR = f'{DRIVE_ROOT}/data/processed/1d_signals_100hz'
META_CSV      = f'{DRIVE_ROOT}/data/processed/metadata/combined_5fold.csv'
LOCAL_SIG_DIR = '/content/signals'    # Colab local disk
WORKERS       = 8

# ─────────────────────────────────────────────────────────────────────────
Path(LOCAL_SIG_DIR).mkdir(parents=True, exist_ok=True)

print('='*60)
print('STAGING signals → /content/signals/')
print('='*60)

df = pd.read_csv(META_CSV, dtype={'id': str}, low_memory=False)
print(f'  CSV rows: {len(df):,}')

total_copied = 0
total_skipped = 0
total_failed = 0
t_start = time.time()

for ds in ['ptbxl', 'samitrop', 'code15']:
    drive_ds = Path(DRIVE_SIG_DIR) / ds
    local_ds = Path(LOCAL_SIG_DIR) / ds
    local_ds.mkdir(parents=True, exist_ok=True)

    ids = df[df['dataset'] == ds]['id'].values
    already = {p.stem for p in local_ds.glob('*.npy')}
    to_copy = [fid for fid in ids if str(fid) not in already]

    print(f'\n{ds}: {len(ids):,} total | {len(already):,} staged | {len(to_copy):,} to copy')

    if not to_copy:
        print(f'  All {ds} signals already staged ✓')
        total_skipped += len(ids)
        continue

    ok = fail = 0
    errors = []

    def copy_one(fid):
        src = drive_ds / f'{fid}.npy'
        dst = local_ds  / f'{fid}.npy'
        try:
            shutil.copy2(str(src), str(dst))
            return 'ok'
        except Exception as e:
            return f'fail:{e}'

    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = {ex.submit(copy_one, fid): fid for fid in to_copy}
        pbar = tqdm(as_completed(futs), total=len(to_copy),
                    desc=f'  {ds}', unit='file', leave=True)
        for fut in pbar:
            res = fut.result()
            if res == 'ok':
                ok += 1
            else:
                fail += 1
                errors.append(f'{futs[fut]}: {res}')
            if ok % 10000 == 0 and ok > 0:
                elapsed = time.time() - t_start
                rate    = ok / max(elapsed, 1)
                pbar.set_postfix({'ok': ok, 'fail': fail,
                                  'rate': f'{rate:.0f}/s'})

    total_copied  += ok
    total_failed  += fail
    if errors[:3]:
        print(f'  Sample errors: {errors[:3]}')

elapsed = time.time() - t_start
print(f'\n{"="*60}')
print(f'Done in {elapsed/60:.1f} min')
print(f'  Staged:  {total_copied:,}')
print(f'  Skipped: {total_skipped:,}  (already on disk)')
print(f'  Failed:  {total_failed:,}')
if total_failed > 0:
    print(f'  ⚠️  {total_failed} failures — re-run this cell to retry')
else:
    print(f'  ✓ All signals on local disk')


## Cell 4 — Configuration

In [ ]:
import os, sys
from pathlib import Path

# ── Drive paths ──────────────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/ChagaSight'
project_root = Path(DRIVE_ROOT)

# Add source directories to Python path
for p in [str(project_root), str(project_root / 'src'),
          str(project_root / 'external' / 'official_2025')]:
    if p not in sys.path:
        sys.path.insert(0, p)

DATA_DIR       = project_root / 'data' / 'processed'
METADATA_CSV   = DATA_DIR / 'metadata' / 'combined_5fold.csv'
IMAGES_DIR     = DATA_DIR / '2d_images'        # read from Drive (direct path, no listing)
SIGNALS_DIR    = Path('/content/signals')       # staged to local disk
CHECKPOINT_DIR = project_root / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MAE_CHECKPOINT   = CHECKPOINT_DIR / 'mae_2d_pretrained.pt'
STMEM_CHECKPOINT = CHECKPOINT_DIR / 'stmem_1d_pretrained.pt'

assert METADATA_CSV.exists(),     f'Missing: {METADATA_CSV}'
assert MAE_CHECKPOINT.exists(),   f'Missing: {MAE_CHECKPOINT}'
assert STMEM_CHECKPOINT.exists(), f'Missing: {STMEM_CHECKPOINT}'
assert (SIGNALS_DIR / 'code15').exists(), f'Stage signals first (Cell 3)'

# ── FOLD ─────────────────────────────────────────────────────────────────────
FOLD = 0   # ← Change to 1, 2, 3, 4 for other folds

# ── QUICK TEST ───────────────────────────────────────────────────────────────
QUICK_TEST = False  # True = 5-min smoke test

# ── HARDWARE ─────────────────────────────────────────────────────────────────
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

# T4 (15GB) or A100 (40GB): both phases can use accum=4 → eff.batch=64
# This exactly matches Van Santvliet → only 12k Phase 2 iters needed (not 24k)
BATCH_SIZE        = 16
PHASE1_GRAD_ACCUM = 4     # eff.batch = 64  ✓ paper
PHASE2_GRAD_ACCUM = 4     # eff.batch = 64  ✓ paper (T4 has 15GB, fits fine)
NUM_WORKERS       = 4     # Colab has more CPU cores
USE_AMP           = True

# A100 bonus: can double batch size for 2× speed
if vram_gb >= 35:
    BATCH_SIZE        = 32
    PHASE1_GRAD_ACCUM = 2   # still eff.batch=64
    PHASE2_GRAD_ACCUM = 2
    print(f'A100 detected ({vram_gb:.0f}GB) — BATCH_SIZE=32 for 2× speed')

# ── TRAINING SCHEDULE ────────────────────────────────────────────────────────
# T4 with accum=4 → eff.batch=64 matches paper exactly → 12k iters (not 24k!)
# Van Santvliet: 12000 iters × eff.batch=64 = 768,000 effective samples
PHASE1_ITERATIONS = 2000   # paper: 2000
PHASE2_ITERATIONS = 12000  # paper: 12000 — correct on Colab because eff.batch=64

PHASE1_LR      = 2e-4
PHASE2_LR_HIGH = 2e-4
PHASE2_LR_LOW  = 2e-5
MAX_GRAD_NORM  = 1.0
WARMUP_ITERS   = 200

# ── VALIDATION ───────────────────────────────────────────────────────────────
# 12000 / 4000 = 3 mid-Phase-2 checks (at 4000, 8000, 12000)
VAL_EVERY = 4000

# ── AUGMENTATION (paper-aligned) ─────────────────────────────────────────────
PAPER_AUGMENTATION_CONFIG = {
    'lead_mixup':       {'prob': 0.3, 'alpha': 0.2},
    'powerline_noise':  {
        'prob': 0.5, 'use_snr': True, 'snr_range': (15, 30),
        'random_freq': True, 'add_harmonics': True,
    },
    'random_shift':     {'prob': 0.5, 'max_shift': 100},
    'amplitude_scaling':{'prob': 0.3, 'scale_range': (0.8, 1.2)},
    'baseline_wander':  {'prob': 0.2, 'amplitude': 0.2, 'freq_range': (0.1, 0.5)},
}

# ── QUICK TEST OVERRIDES ─────────────────────────────────────────────────────
if QUICK_TEST:
    PHASE1_ITERATIONS = 25
    PHASE2_ITERATIONS = 50
    WARMUP_ITERS      = 10
    VAL_EVERY         = 50
    NUM_WORKERS       = 0

# ── AUTO-RESUME ──────────────────────────────────────────────────────────────
_ckpt = CHECKPOINT_DIR / f'fold{FOLD}_latest.pt'
RESUME_FROM = str(_ckpt) if _ckpt.exists() else None

# ── SUMMARY ──────────────────────────────────────────────────────────────────
p1 = BATCH_SIZE * PHASE1_GRAD_ACCUM * PHASE1_ITERATIONS
p2 = BATCH_SIZE * PHASE2_GRAD_ACCUM * PHASE2_ITERATIONS
paper_p2 = 64 * 12000
match = '✓ matches paper' if p2 == paper_p2 else f'✗ paper={paper_p2:,}'

print(f'Fold {FOLD}  |  QUICK_TEST={QUICK_TEST}')
print(f'GPU: {vram_gb:.0f} GB  |  Batch {BATCH_SIZE}')
print(f'P1 eff.batch={BATCH_SIZE*PHASE1_GRAD_ACCUM} | P2 eff.batch={BATCH_SIZE*PHASE2_GRAD_ACCUM}')
print(f'Phase 1: {PHASE1_ITERATIONS} iters → {p1:,} eff-samples')
print(f'Phase 2: {PHASE2_ITERATIONS} iters → {p2:,} eff-samples  {match}')
print(f'VAL_EVERY={VAL_EVERY}  |  Resume: {RESUME_FROM or "fresh start"}')
if not QUICK_TEST:
    eta_p1 = 2000 * 0.35 / 60  # ~12 min on T4
    eta_p2 = PHASE2_ITERATIONS * (0.35 if vram_gb < 20 else 0.12) / 60
    print(f'ETA: Phase 1 ~{eta_p1:.0f} min  |  Phase 2 ~{eta_p2/60:.1f} h  |  Total ~{(eta_p1+eta_p2)/60:.1f} h')


## Cell 5 — Imports + Keepalive

In [ ]:
import torch
import numpy as np
import pandas as pd
import threading, time
from pathlib import Path
from datetime import datetime

from src.models.hybrid_model import HybridChagasModel
from src.training.dataset    import create_dataloaders, ChagasDataset, custom_collate_fn
from src.training.trainer    import ChagasTrainer
from torch.utils.data        import DataLoader, WeightedRandomSampler

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise RuntimeError('No GPU — Runtime > Change runtime type > T4 GPU')

print(f'✓ Device: {torch.cuda.get_device_name(0)}')

# ── Keepalive — prevents Colab idle disconnect ────────────────────────────────
def _keepalive(interval=1800):
    count = 0
    while True:
        time.sleep(interval)
        count += 1
        print(f'  ♥ Keepalive #{count}  {datetime.now().strftime("%H:%M:%S")} '
              f'| VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

threading.Thread(target=_keepalive, daemon=True).start()
print('Keepalive thread started (heartbeat every 30 min)')


## Cell 6 — GPU Memory Check

In [ ]:
torch.cuda.empty_cache()
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
free_gb  = total_gb - torch.cuda.memory_reserved() / 1e9
print(f'VRAM: {total_gb:.1f} GB total | {free_gb:.2f} GB free')
if free_gb < 3.5:
    raise RuntimeError(f'Only {free_gb:.1f} GB free — restart runtime (Runtime > Restart)')
print('Memory check passed.')


## Cell 7 — Dataloaders

Signals are loaded from `/content/signals/` (local disk, fast).
Images are loaded from Drive by direct path (no directory listing = FUSE OK).


In [ ]:
# Create datasets manually so we can point images→Drive, signals→local disk
train_dataset = ChagasDataset(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),       # Drive — direct file opens, no listing
    signals_dir=str(SIGNALS_DIR),     # /content/signals/ — local disk, fast
    split='train', fold=FOLD,
    augment=True,
    augmentation_config=PAPER_AUGMENTATION_CONFIG,
    use_soft_labels=True,
)
val_dataset = ChagasDataset(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),
    signals_dir=str(SIGNALS_DIR),
    split='val', fold=FOLD,
    augment=False,
    use_soft_labels=True,
)

# Weighted sampler: 5× oversample positives (Van Santvliet 2025)
sample_weights = train_dataset.get_sample_weights()
train_sampler  = WeightedRandomSampler(sample_weights, len(train_dataset), replacement=True)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
    collate_fn=custom_collate_fn,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
    collate_fn=custom_collate_fn,
)

# Sanity check
batch = next(iter(train_loader))
assert batch['image'].shape  == torch.Size([BATCH_SIZE, 3, 24, 2048])
assert batch['signal'].shape == torch.Size([BATCH_SIZE, 12, 1000])
assert not torch.isnan(batch['signal']).any(), 'NaN in signals'
unique_labels = sorted({round(v, 1) for v in batch['label'].tolist()})
assert set(unique_labels) <= {0.0, 0.2, 0.8, 1.0}, f'Bad labels: {unique_labels}'

n_train   = len(train_dataset)
n_pos     = int(train_dataset.df['label_hard'].sum())
n_val     = len(val_dataset)
n_pos_val = int(val_dataset.df['label_hard'].sum())
print(f'Train: {n_train:,} ({n_pos:,} pos = {100*n_pos/n_train:.2f}%)')
print(f'Val:   {n_val:,}  ({n_pos_val:,} pos = {100*n_pos_val/n_val:.2f}%)')
print(f'Label values in batch: {unique_labels}')
print(f'Batches per epoch: train={len(train_loader):,}  val={len(val_loader):,}')


## Cell 8 — Model + Pretrained Weights

In [ ]:
model = HybridChagasModel(
    img_size=(24, 2048), patch_size_2d=(8, 64),
    num_leads=12, seq_len_1d=1000, patch_size_1d=50,
    embed_dim=768, depth=12, num_heads=12,
    use_aol=True, use_demographics=True,
)

# flatten_dict fix: loads all 145+ transformer weights correctly
model.vit_2d.load_mae_pretrained(str(MAE_CHECKPOINT))
model.vit_1d_fm.load_stmem_pretrained(str(STMEM_CHECKPOINT))

model = model.to(device)

with torch.no_grad():
    out = model(batch['image'].to(device), batch['signal'].to(device),
                batch['age'].to(device),   batch['sex'].to(device))
assert torch.isfinite(out['logits']).all()
assert torch.isfinite(out['fm_features']).all()

total_p = sum(p.numel() for p in model.parameters())
print(f'Params: {total_p:,}  |  logits: {out["logits"].shape}  |  FM: {out["fm_features"].shape}')
print(f'VRAM after model load: {torch.cuda.memory_allocated()/1e9:.1f} GB used')


## Cell 9 — Trainer

In [ ]:
trainer = ChagasTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    phase1_iterations=PHASE1_ITERATIONS,   # 2,000
    phase2_iterations=PHASE2_ITERATIONS,   # 12,000 (eff.batch=64 matches paper)
    phase1_lr=PHASE1_LR,
    phase2_lr_high=PHASE2_LR_HIGH,
    phase2_lr_low=PHASE2_LR_LOW,
    checkpoint_dir=str(CHECKPOINT_DIR),    # saves to Drive automatically
    use_amp=USE_AMP,
    max_grad_norm=MAX_GRAD_NORM,
    warmup_iters=WARMUP_ITERS,
    phase1_grad_accum=PHASE1_GRAD_ACCUM,   # 4 → eff.batch 64
    phase2_grad_accum=PHASE2_GRAD_ACCUM,   # 4 → eff.batch 64  ✓ paper
    val_every_n_iters=VAL_EVERY,           # 4,000
    val_subset_size=300  if QUICK_TEST else 3000,
    val_n_permutations=100 if QUICK_TEST else 1000,
)

p2_checks = PHASE2_ITERATIONS // VAL_EVERY
print(f'Val every {VAL_EVERY} iters | P1 checks: 0 | P2 checks: {p2_checks}')
print(f'Val subset: {trainer.val_subset_size} | Perms: {trainer.val_n_permutations}')

# Effective batch verification
eff_p1 = BATCH_SIZE * PHASE1_GRAD_ACCUM
eff_p2 = BATCH_SIZE * PHASE2_GRAD_ACCUM
paper  = 64 * 12000
our    = BATCH_SIZE * PHASE2_GRAD_ACCUM * PHASE2_ITERATIONS
match  = '✓ matches paper' if our == paper else f'✗ paper={paper:,}'
print(f'P1 eff.batch={eff_p1}  P2 eff.batch={eff_p2}')
print(f'Phase 2 total eff-samples: {our:,}  {match}')


## Cell 10 — Train

**After a disconnect**: re-run Cells 1 → 3 → 4 → 5 → 7 → 8 → 9 → this cell.
Training resumes automatically from the last checkpoint saved to Drive.


In [ ]:
if RESUME_FROM:
    print(f'Resuming from: {Path(RESUME_FROM).name}')
else:
    print('Starting fresh')

metrics = trainer.train(fold=FOLD, resume_from=RESUME_FROM)

tpr = metrics['tpr_5pct']
print(f'\nFold {FOLD} complete:')
print(f'  TPR@5%: {tpr:.4f}  (PRIMARY — official PhysioNet metric)')
print(f'  AUROC:  {metrics["auroc"]:.4f}')
print(f'  AUPRC:  {metrics.get("auprc", 0):.4f}')
print(f'  Method: {"OFFICIAL" if metrics.get("using_official") else "APPROXIMATE"}')

if not QUICK_TEST:
    for name, val in [
        ('Random baseline',             0.050),
        ('Kim 2025 hidden val',          0.369),
        ('Challenge target',            0.420),
        ('Van Santvliet val (top team)', 0.445),
        ('Van Santvliet CV mean',        0.490),
    ]:
        diff = tpr - val
        print(f'  {"↑" if diff >= 0 else "↓"}{abs(diff):.4f}  vs  {name} ({val:.3f})')


## Cell 11 — Save Results & Training Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

results_df = pd.DataFrame([metrics])
results_df['fold'] = FOLD
results_df['phase2_iters'] = PHASE2_ITERATIONS
results_df['phase2_eff_batch'] = BATCH_SIZE * PHASE2_GRAD_ACCUM
results_df.to_csv(CHECKPOINT_DIR / f'fold{FOLD}_results.csv', index=False)
print(f'Results saved: fold{FOLD}_results.csv')

history = trainer.history
fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3)

ax0 = fig.add_subplot(gs[0])
if history['train_loss']:
    ax0.plot(history['train_loss'], lw=0.8, alpha=0.8, color='steelblue')
    if PHASE1_ITERATIONS < len(history['train_loss']):
        ax0.axvline(PHASE1_ITERATIONS, color='r', ls='--', lw=1.2,
                    label=f'Phase 2 start ({PHASE1_ITERATIONS})')
        ax0.legend(fontsize=8)
    ax0.set(xlabel='Iteration', ylabel='Loss', title='Training Loss')
    ax0.grid(True, alpha=0.3)

ax1 = fig.add_subplot(gs[1])
if history['val_tpr_5pct']:
    iters = [VAL_EVERY * (i + 1) for i in range(len(history['val_tpr_5pct']))]
    ax1.plot(iters, history['val_tpr_5pct'], 'go-', ms=5, lw=1.5)
    ax1.axhline(0.420, color='r',      ls='--', lw=1, label='Target 0.420')
    ax1.axhline(0.445, color='purple', ls=':',  lw=1, label='Van Santvliet 0.445')
    ax1.set(xlabel='Iteration', ylabel='TPR@5%', title='Val TPR@5%')
    ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[2])
if history['grad_norm']:
    g = history['grad_norm'][:PHASE1_ITERATIONS]
    ax2.plot(g, lw=0.6, alpha=0.6, color='darkorange')
    ax2.axhline(1.0, color='r', ls='--', lw=1.2, label='Clip @ 1.0')
    clipped = 100 * sum(1 for v in g if v > 1.0) / max(1, len(g))
    ax2.set(xlabel='Iteration', ylabel='Grad norm',
            title=f'Grad Norm Phase 1 ({clipped:.0f}% clipped)')
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

fig.suptitle(
    f'Fold {FOLD} | Colab | P2 iters={PHASE2_ITERATIONS} | eff.batch={BATCH_SIZE*PHASE2_GRAD_ACCUM}',
    fontsize=12, fontweight='bold')
plt.tight_layout()
path = CHECKPOINT_DIR / f'fold{FOLD}_training_curve.png'
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {path.name}  (in Drive/ChagaSight/checkpoints/)')


## Cell 12 — Checkpoint Verification

In [ ]:
print(f'Checkpoints for fold {FOLD}:')
for p in sorted(CHECKPOINT_DIR.glob(f'fold{FOLD}*.pt')):
    mb = p.stat().st_size / 1e6
    print(f'  {p.name:<40}  {mb:.0f} MB')

best = CHECKPOINT_DIR / f'fold{FOLD}_best.pt'
if best.exists():
    ckpt = torch.load(best, map_location='cpu', weights_only=False)
    vs = ckpt.get('val_score', None)
    score_str = f'{vs:.4f}' if vs is not None else 'n/a'
    print(f'\nbest:  val_score={score_str}  phase={ckpt.get("phase","?")}  iter={ckpt.get("iteration","?")}')
    del ckpt

# All-fold status
print('\nAll-fold status:')
for f in range(5):
    done = (CHECKPOINT_DIR / f'fold{f}_best.pt').exists()
    score = ''
    if done:
        try:
            c = torch.load(CHECKPOINT_DIR / f'fold{f}_best.pt', map_location='cpu', weights_only=False)
            vs = c.get('val_score', None)
            score = f'  TPR@5%={vs:.4f}' if vs else ''
            del c
        except:
            pass
    print(f'  [{"✓" if done else "○"}] Fold {f}{score}')

if all((CHECKPOINT_DIR / f'fold{f}_best.pt').exists() for f in range(5)):
    print('\n✓ All 5 folds complete — run evaluation_complete_v3.1.ipynb')
else:
    nxt = next((f for f in range(5) if not (CHECKPOINT_DIR / f'fold{f}_best.pt').exists()), None)
    if nxt is not None:
        print(f'\nNext: change FOLD={nxt} in Cell 4 and re-run from Cell 7')

# Remind where files are
print(f'\nAll checkpoints saved to Google Drive:')
print(f'  {CHECKPOINT_DIR}')
print(f'  Download fold{{n}}_best.pt to your local machine after all 5 folds are done.')
